# Senate: does fundraising help once polls and ratings are blended in?

In Senate Stage 1 alone (2012–2022), adding `fundraising_advantage` gave the best ex-ante competitive MAE (5.90 vs
6.56 for S1). But it was too Democratic (bias −3.2), it absorbed the lean and generic-ballot coefficients, and its FEC
totals include post-election money. In the Senate, polls and expert ratings are blended in after Stage 1 and already
carry candidate strength, so this notebook tests whether any gain survives the blend.

- **S1:** production, `partisan_lean + gen_margin_raw + 2.5 × inc_signed`.
- **S1 + fundraising:** the same plus `fundraising_advantage`. It's defined here only; production doesn't change.
- **Both:** equal-year WLS on expanding windows (each year fit on earlier ones), blended with the current Stage 2
  weights (0.37 / 0.54 / 0.09 with polls, 0.65 / 0 / 0.35 without). Scored on the same 166 races, 2016–2024.

**Caveat:** historical `fundraising_advantage` uses FEC totals through December 31 of the election year, so it
includes some post-election money. Any gain below is an upper bound on what a clean pre-election measure would give.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from senate import stage1, stage2

pd.set_option("display.width", 200)
EVAL_YEARS = [2016, 2018, 2020, 2022, 2024]  # the years with expert ratings
S1_FUND = stage1.Stage1Config("senate_s1_fundraising",
                              f"{sr.TARGET} ~ partisan_lean + gen_margin_raw + fundraising_advantage",
                              fixed={"inc_signed": stage1.FIXED_INC})
CONFIGS = {"S1": stage1.SENATE_S1, "S1 + fundraising": S1_FUND}

data = sr.load_data()
polls, ratings = stage2.load_inputs()
assert not data.loc[data["observed"], "fundraising_advantage"].isna().any()
rolling = {name: stage1.rolling_predictions(c, eval_years=EVAL_YEARS, data=data) for name, c in CONFIGS.items()}
blends = {name: stage2.blend(pd.concat([r.predictions for r in runs.values()], ignore_index=True), polls, ratings)
          for name, runs in rolling.items()}
keys = [b[sr.KEY].apply(tuple, axis=1).tolist() for b in blends.values()]
assert keys[0] == keys[1], "the two specifications must score the same races"
print(len(keys[0]), "races scored by both")

166 races scored by both


## 1. Fundamentals alone and in the blend (pooled 2016–2024)

In [2]:
COLUMNS = {"fundamentals only": "margin_fundamentals", "blend": "prediction"}
pd.concat({name: stage2.score_table(b, data, COLUMNS) for name, b in blends.items()}, names=["stage1"]).round(2)

n   mae  bias  ex_ante_n  ex_ante_mae  accuracy_pct  ex_ante_accuracy_pct  bias_inc_R  bias_inc_open  bias_inc_D
stage1                                                                                                                                                  
S1               fundamentals only  166.0  6.01 -1.11       51.0         4.98         92.17                 80.39       -3.92          -1.34        1.52
                 blend              166.0  4.76 -1.86       51.0         3.58         93.37                 84.31       -3.42          -1.66       -0.55
S1 + fundraising fundamentals only  166.0  6.71 -4.11       51.0         4.51         90.96                 80.39       -3.27          -3.66       -5.08
                 blend              166.0  5.29 -3.19       51.0         3.32         93.98                 86.27       -2.98          -2.67       -3.63

## 2. The blend, polled vs unpolled races

Unpolled races (mostly safe seats) put 0.65 on fundamentals, so that's where fundraising could matter most.

In [3]:
SHOW = ["n", "mae", "bias", "ex_ante_n", "ex_ante_mae", "accuracy_pct"]
for polled in [True, False]:
    display(Markdown(f"**{'Polled' if polled else 'Unpolled'} races**"))
    display(pd.concat({name: stage2.score_table(b[b["polling_available"] == polled], data, {"blend": "prediction"})
                       for name, b in blends.items()}, names=["stage1"])[SHOW].round(2))

**Polled races**

,,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct
stage1,,,,,,,
S1,blend,109.0,4.35,-1.92,48.0,3.65,89.91
S1 + fundraising,blend,109.0,4.56,-3.15,48.0,3.41,91.74


**Unpolled races**

,,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct
stage1,,,,,,,
S1,blend,57.0,5.54,-1.75,3.0,2.52,100.00
S1 + fundraising,blend,57.0,6.70,-3.28,3.0,1.98,98.25


## 3. By year (blend MAE, all races and ex-ante competitive)

In [4]:
by_year = {}
for name, b in blends.items():
    rows = b.merge(data[[*sr.KEY, "partisan_lean", "gb_margin_2p_norm"]], on=sr.KEY)
    ex_ante = (rows["partisan_lean"] + rows["gb_margin_2p_norm"]).abs() <= sr.EX_ANTE_THRESHOLD
    err = rows["residual"].abs()
    by_year[(name, "all")] = err.groupby(rows["year"]).mean()
    by_year[(name, "ex_ante")] = err[ex_ante].groupby(rows.loc[ex_ante, "year"]).mean()
pd.DataFrame(by_year).T.round(2)

year                      2016  2018  2020  2022  2024
S1               all      7.05  3.38  5.13  4.28  4.06
                 ex_ante  5.05  3.01  2.93  3.94  2.76
S1 + fundraising all      6.63  3.80  6.98  4.33  4.76
                 ex_ante  4.47  2.68  3.73  2.70  2.74

## 4. Coefficients by window (Stage 1)

In [5]:
pd.concat({name: pd.DataFrame({y: r.coefficients for y, r in runs.items()}).T for name, runs in rolling.items()},
          names=["stage1", "eval_year"]).round(2)

Intercept  partisan_lean  gen_margin_raw  inc_signed  fundraising_advantage
stage1           eval_year                                                                             
S1               2016            2.36           0.95            0.83         2.5                    NaN
                 2018            1.64           0.99            0.84         2.5                    NaN
                 2020            1.61           0.96            0.87         2.5                    NaN
                 2022            1.43           0.97            0.85         2.5                    NaN
                 2024            1.69           0.97            0.83         2.5                    NaN
S1 + fundraising 2016            1.69           0.49            0.59         2.5                   6.34
                 2018            0.87           0.51            0.61         2.5                   6.04
                 2020            0.54           0.52            0.56         2.5                   5.78
                 2022            0.12           0.57            0.51         2.5                   5.41
                 2024           -0.02           0.58            0.55         2.5                   5.02

## Conclusion

**No benefit once polls and ratings are blended in.** Fundraising slightly improves ex-ante competitive races, but
makes everything else worse.

| | S1 blend | S1 + fundraising blend |
|---|---|---|
| MAE, all 166 races | 4.76 | 5.29 |
| Bias | −1.86 | −3.19 |
| Ex-ante competitive MAE (51) | 3.58 | 3.32 |
| Winners, ex-ante competitive | 84.3% | 86.3% |
| MAE, unpolled races (57) | 5.54 | 6.70 |

- **The overall MAE worsens by 0.5 points.** Most of the damage is in unpolled races, which lean on fundamentals and
  where fundraising adds about 1.2 points of error.
- **The bias nearly doubles,** toward Democrats. Democratic incumbents are hit hardest: bias −0.6 becomes −3.6.
- **The competitive-race gain is small:** 0.26 points of MAE and 1 more correct call in 51 races. It comes with the
  FEC look-ahead caveat, so a clean pre-election measure would likely gain less.
- **Recommendation:** keep fundraising out of Senate Stage 1. If it's revisited, use FEC totals cut at a
  pre-election date, and test it only for the competitive or polled races.